# Export LCF data to Excel for browsing

This notebook loads each year's LCF Stata files (`dvhh` = household, `dvper` = person) and exports them to Excel files you can open and scroll through.

**These Excel files are for human inspection only** — the production pipeline (`wrangle_lcf.py`) still reads the original `.dta` files. Treat these exports as disposable scratch copies.

## What you get per year

One file named `lcf_<year>_view.xlsx` with four sheets:

| Sheet | Contents |
|---|---|
| `dvhh_data` | All household rows, all columns |
| `dvhh_dictionary` | Every `dvhh` variable code + its label |
| `dvhh_summary` | min/max/mean/std for every numeric column |
| `dvhh_value_counts` | Counts for key categorical columns (a121, gorx, etc.) |
| `dvper_data` | All person rows, all columns |
| `dvper_dictionary` | Every `dvper` variable code + its label |

## Prerequisites

- Raw LCF files present at `data/raw/LCF/LCF_<year>/`
- Python packages: `pandas`, `openpyxl` (already in `requirements.txt`)

## 1. Setup

In [ ]:
import pathlib
import warnings

import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# Project root = parent of this notebook's folder
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
RAW = ROOT / "data" / "raw" / "LCF"
OUTPUT_DIR = ROOT / "data" / "lcf_viewers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw LCF:      {RAW}")
print(f"Output dir:   {OUTPUT_DIR}")

## 2. File paths (same as `wrangle_lcf.py`)

In [ ]:
LCF_DVHH = {
    2015: RAW / "LCF_2015" / "stata" / "stata11_se" / "2015-16_dvhh_ukanon.dta",
    2016: RAW / "LCF_2016" / "stata" / "stata13_se" / "2016_17_dvhh_ukanon.dta",
    2017: RAW / "LCF_2017" / "stata" / "stata11_se" / "dvhh_ukanon_2017-18.dta",
    2018: RAW / "LCF_2018" / "stata" / "stata13"    / "2018_dvhh_ukanon.dta",
    2019: RAW / "LCF_2019" / "stata" / "stata13"    / "lcfs_2019_dvhh_ukanon.dta",
    2020: RAW / "LCF_2020" / "stata" / "stata13"    / "lcfs_2020_dvhh_ukanon.dta",
    2021: RAW / "LCF_2021" / "stata" / "stata13_se" / "lcfs_2021_dvhh_ukanon.dta",
    2022: RAW / "LCF_2022" / "stata" / "stata13_se" / "dvhh_ukanon_2022.dta",
    2023: RAW / "LCF_2023" / "stata" / "stata13_se" / "dvhh_ukanon_v2_2023.dta",
}

LCF_DVPER = {
    2015: RAW / "LCF_2015" / "stata" / "stata11_se" / "2015-16_dvper_ukanon.dta",
    2016: RAW / "LCF_2016" / "stata" / "stata13_se" / "2016_17_dvper_ukanon.dta",
    2017: RAW / "LCF_2017" / "stata" / "stata11_se" / "dvper_ukanon_2017-18.dta",
    2018: RAW / "LCF_2018" / "stata" / "stata13"    / "2018_dvper_ukanon201819.dta",
    2019: RAW / "LCF_2019" / "stata" / "stata13"    / "lcfs_2019_dvper_ukanon201920.dta",
    2020: RAW / "LCF_2020" / "stata" / "stata13"    / "lcfs_2020_dvper_ukanon202021.dta",
    2021: RAW / "LCF_2021" / "stata" / "stata13_se" / "lcfs_2021_dvper_ukanon202122.dta",
    2022: RAW / "LCF_2022" / "stata" / "stata13_se" / "dvper_ukanon_2022-23.dta",
    2023: RAW / "LCF_2023" / "stata" / "stata13_se" / "dvper_ukanon_202324_2023.dta",
}

# Quick sanity check
for yr, p in LCF_DVHH.items():
    status = "OK" if p.exists() else "MISSING"
    print(f"  {yr} dvhh:  {status}  {p.name}")
print()
for yr, p in LCF_DVPER.items():
    status = "OK" if p.exists() else "MISSING"
    print(f"  {yr} dvper: {status}  {p.name}")

## 3. Helper: load one .dta file and capture its variable labels

Stata's variable labels are the single most useful metadata — they tell you what every cryptic code like `a121` actually means. We extract them into a separate dataframe.

In [ ]:
def load_stata_with_labels(path: pathlib.Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Read a .dta file and return (data, dictionary).

    data       - the dataframe itself with columns lowercased
    dictionary - dataframe with columns [code, label] describing every variable
    """
    # Load the data
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()

    # Separately open a StataReader to pull variable labels
    try:
        reader = pd.io.stata.StataReader(path)
        var_labels = reader.variable_labels()
        reader.close()
    except Exception as e:
        print(f"    (warning: could not read variable labels: {e})")
        var_labels = {}

    # Turn into a dataframe, lowercase codes, sort
    dictionary = (
        pd.DataFrame(list(var_labels.items()), columns=["code", "label"])
        .assign(code=lambda d: d["code"].str.lower())
        .sort_values("code")
        .reset_index(drop=True)
    )

    return df, dictionary

## 4. Export function for one year

This builds one multi-sheet Excel file for a single year.

In [ ]:
# Key categorical columns we want value-counts for.
# You can edit this list to add/remove variables.
CATEGORICAL_COLS = [
    "a121",   # Tenure type 1
    "a122",   # Tenure type 2
    "gorx",   # Government Office Region
    "a065p",  # HRP age band (coded)
    "a062",   # Family type
    "a049",   # Household size
    "sexhrp", # Sex of HRP
]


def export_year_to_excel(year: int) -> pathlib.Path:
    """Build lcf_<year>_view.xlsx in OUTPUT_DIR."""
    dvhh_path = LCF_DVHH[year]
    dvper_path = LCF_DVPER[year]

    print(f"[{year}] loading dvhh...")
    dvhh, dvhh_dict = load_stata_with_labels(dvhh_path)
    print(f"        {len(dvhh):,} households x {dvhh.shape[1]} columns")

    print(f"[{year}] loading dvper...")
    dvper, dvper_dict = load_stata_with_labels(dvper_path)
    print(f"        {len(dvper):,} persons x {dvper.shape[1]} columns")

    # Summary stats for all numeric cols in dvhh
    summary = dvhh.describe(include="number").T
    summary = summary[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    summary.index.name = "code"
    # join variable label for readability
    summary = summary.reset_index().merge(dvhh_dict, on="code", how="left")

    # Value counts for key categorical columns
    vc_rows = []
    for col in CATEGORICAL_COLS:
        if col not in dvhh.columns:
            continue
        counts = dvhh[col].value_counts(dropna=False).sort_index()
        for val, n in counts.items():
            vc_rows.append({"column": col, "value": val, "count": int(n)})
    value_counts_df = pd.DataFrame(vc_rows)

    out_path = OUTPUT_DIR / f"lcf_{year}_view.xlsx"
    print(f"[{year}] writing {out_path.name}...")
    with pd.ExcelWriter(out_path, engine="openpyxl") as w:
        dvhh.to_excel(w, sheet_name="dvhh_data", index=False)
        dvhh_dict.to_excel(w, sheet_name="dvhh_dictionary", index=False)
        summary.to_excel(w, sheet_name="dvhh_summary", index=False)
        if not value_counts_df.empty:
            value_counts_df.to_excel(w, sheet_name="dvhh_value_counts", index=False)
        dvper.to_excel(w, sheet_name="dvper_data", index=False)
        dvper_dict.to_excel(w, sheet_name="dvper_dictionary", index=False)

    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"[{year}] done ({size_mb:.1f} MB)\n")
    return out_path

## 5. Export all 9 years

Typical runtime: ~2-4 minutes total. Each file will be 40-80 MB (dvhh has ~500-600 columns × 5000 rows).

In [ ]:
# Edit this list if you only want specific years
YEARS_TO_EXPORT = list(range(2015, 2024))

exported = []
for yr in YEARS_TO_EXPORT:
    try:
        out = export_year_to_excel(yr)
        exported.append(out)
    except Exception as e:
        print(f"[{yr}] FAILED: {e}\n")

print(f"\nAll done. Exported {len(exported)} / {len(YEARS_TO_EXPORT)} years.")

## 6. Verify outputs

In [ ]:
print(f"Files in {OUTPUT_DIR}:\n")
for p in sorted(OUTPUT_DIR.glob("*.xlsx")):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:30s} {size_mb:6.1f} MB")

## Optional: combined cross-year variable dictionary

Handy for spotting which variables exist in which years (schema changes year-to-year).

In [ ]:
all_dicts = []
for yr in YEARS_TO_EXPORT:
    try:
        _, dvhh_dict = load_stata_with_labels(LCF_DVHH[yr])
        dvhh_dict["year"] = yr
        dvhh_dict["file"] = "dvhh"
        all_dicts.append(dvhh_dict)

        _, dvper_dict = load_stata_with_labels(LCF_DVPER[yr])
        dvper_dict["year"] = yr
        dvper_dict["file"] = "dvper"
        all_dicts.append(dvper_dict)
    except Exception as e:
        print(f"[{yr}] FAILED: {e}")

combined = pd.concat(all_dicts, ignore_index=True)

# Pivot: one row per (code, file), one column per year, value = label if present
pivot = (
    combined.pivot_table(
        index=["file", "code"],
        columns="year",
        values="label",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["file", "code"])
)

master_path = OUTPUT_DIR / "lcf_variable_dictionary_all_years.xlsx"
pivot.to_excel(master_path, index=False)
print(f"Saved: {master_path}")
print(f"Shape: {pivot.shape[0]} unique variables across {pivot.shape[1] - 2} years")

## Using the viewer files

Open any `lcf_<year>_view.xlsx` in Excel. Suggested workflow:

1. Start with **`dvhh_dictionary`** — scroll through to see what's in the file. Ctrl+F for words like "tenure", "income", "expenditure".
2. Go to **`dvhh_value_counts`** — see how many households fall into each category of the key variables.
3. Go to **`dvhh_summary`** — see min/max/mean for every numeric variable. Extreme values here hint at cleaning issues.
4. Finally **`dvhh_data`** — browse actual household rows. Use Ctrl+Right to jump across columns.
5. Open **`lcf_variable_dictionary_all_years.xlsx`** to see which variables exist in each year — empty cells flag schema changes.

These files are **disposable**. If your LCF data updates, re-run this notebook to regenerate them. Don't commit them to git — they're large and reconstructable.